In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Load Dataset

In [2]:
TRAIN_PATH = "/kaggle/input/WiDSWorldWide_GlobalDathon26/train.csv"
TEST_PATH  = "/kaggle/input/WiDSWorldWide_GlobalDathon26/test.csv"
META_DATA  = "/kaggle/input/WiDSWorldWide_GlobalDathon26/metaData.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
meta = pd.read_csv(META_DATA)


print("Train shape:", train.shape)
print("Test  shape:", test.shape)
display(train.head())

Train shape: (221, 37)
Test  shape: (95, 35)


,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


# Dataset Overview

In [3]:
summary = pd.DataFrame({
    "column": train.columns,
    "dtype": train.dtypes,
    "missing_values": train.isnull().sum(),
    "unique_values": train.nunique()
})

display(summary)

,column,dtype,missing_values,unique_values
event_id,event_id,int64,0,221
num_perimeters_0_5h,num_perimeters_0_5h,int64,0,12
dt_first_last_0_5h,dt_first_last_0_5h,float64,0,62
low_temporal_resolution_0_5h,low_temporal_resolution_0_5h,int64,0,2
area_first_ha,area_first_ha,float64,0,221
area_growth_abs_0_5h,area_growth_abs_0_5h,float64,0,26
area_growth_rel_0_5h,area_growth_rel_0_5h,float64,0,26
area_growth_rate_ha_per_h,area_growth_rate_ha_per_h,float64,0,26
log1p_area_first,log1p_area_first,float64,0,221
log1p_growth,log1p_growth,float64,0,25


# Target Variable Exploration

In [4]:
fig = px.pie(
    train,
    names="event",
    title="Fire Reaching Evacuation Zone within 72h",
)

fig.show()

In [5]:
fig = px.histogram(train, x="event", color="event", 
                   title="Distribution of Event Variable")
fig.show()

## Time to Hit Distribution

In [6]:
fig = px.histogram(
    train,
    x="time_to_hit_hours",
    color="event",
    nbins=221,
    title="Distribution of Time to Hit Evacuation Zone"
)

fig.show()

# Class Imbalance Visualization

In [7]:
counts = train['event'].value_counts().reset_index()
counts.columns = ['event','count']

fig = px.bar(
    counts,
    x="event",
    y="count",
    text="count",
    title="Class Imbalance"
)

fig.show()

In [8]:
# #low_temporal_resolution_0_5h
# counts = train['low_temporal_resolution_0_5h'].value_counts().reset_index()
# counts.columns = ['low_temporal_resolution_0_5h','count']

# fig = px.bar(
#     counts,
#     x="low_temporal_resolution_0_5h",
#     y="count",
#     text="count",
#     title="Class Imbalance",
# )

# fig.show()

fig = px.histogram(
    train,
    x="low_temporal_resolution_0_5h",
    color="event",
    nbins=2,
    title="Distribution of low_temporal_resolution_0_5h"
)

fig.show()

# Growth Feature Exploration

In [9]:
growth_features = [
"area_first_ha",
"area_growth_abs_0_5h",
"area_growth_rate_ha_per_h",
"radial_growth_m",
"radial_growth_rate_m_per_h"
]

for col in growth_features:

    fig = px.histogram(
        train,
        x=col,
        color="event",
        marginal="box",
        title=f"Distribution of {col}"
    )

    fig.show()

# Fire Size vs Growth Scatter

In [10]:
train["area_growth_rate_ha_per_h"] = np.where(train["area_growth_rate_ha_per_h"]<0, 0, train["area_growth_rate_ha_per_h"])
train["area_first_ha"] = np.log(train["area_first_ha"])

In [11]:
fig = px.scatter(
    train,
    x="area_first_ha",
    y="area_growth_abs_0_5h",
    color="event",
    size="area_growth_rate_ha_per_h",
    hover_data=["time_to_hit_hours"],
    title="Initial Fire Size vs Growth"
)

fig.show()

# Distance to Evacuation Zone Analysis

In [12]:
dist_features = [
"dist_min_ci_0_5h",
"closing_speed_m_per_h",
"projected_advance_m",
"dist_slope_ci_0_5h"
]

for col in dist_features:

    fig = px.box(
        train,
        x="event",
        y=col,
        title=f"{col} vs Event Outcome"
    )

    fig.show()

# Distance Change Over Time

In [13]:
train["projected_advance_m"] = train["projected_advance_m"] - train["projected_advance_m"].min()

In [14]:
fig = px.scatter(
    train,
    x="dist_change_ci_0_5h",
    y="closing_speed_m_per_h",
    color="event",
    size="projected_advance_m",
    title="Distance Change vs Closing Speed"
)

fig.show()

# Fire Direction Analysis

In [15]:
fig = px.scatter(
    train,
    x="spread_bearing_sin",
    y="spread_bearing_cos",
    color="event",
    title="Fire Spread Direction"
)

fig.show()

# Alignment with Evacuation Direction

In [16]:
fig = px.histogram(
    train,
    x="alignment_abs",
    color="event",
    nbins=30,
    title="Alignment Between Fire Spread and Evacuation Direction"
)

fig.show()

# Temporal Patterns

In [17]:
fig = px.histogram(
    train,
    x="event_start_hour",
    color="event",
    nbins=24,
    title="Fire Start Hour Distribution"
)

fig.show()

# Fires by Month

In [18]:
fig = px.histogram(
    train,
    x="event_start_month",
    color="event",
    title="Wildfires by Month"
)

fig.show()

# Correlation Matrix

In [19]:
numeric_cols = train.select_dtypes(include=np.number).columns

corr = train[numeric_cols].corr()

fig = px.imshow(
    corr,
    color_continuous_scale="RdBu",
    title="Feature Correlation Matrix"
)

fig.show()

# Feature Importance Proxy (EDA)

In [20]:
corr_target = corr["event"].abs().sort_values(ascending=False)

corr_target.head(15)

event                           1.000000
time_to_hit_hours               0.719485
dist_min_ci_0_5h                0.481379
low_temporal_resolution_0_5h    0.379117
num_perimeters_0_5h             0.370501
dt_first_last_0_5h              0.352954
alignment_abs                   0.349115
spread_bearing_cos              0.323189
log1p_growth                    0.292688
spread_bearing_deg              0.281012
log_area_ratio_0_5h             0.229327
radial_growth_rate_m_per_h      0.214956
radial_growth_m                 0.209343
centroid_speed_m_per_h          0.209254
centroid_displacement_m         0.207992
Name: event, dtype: float64

In [21]:
top_corr = corr_target[1:15]

fig = px.bar(
    x=top_corr.values,
    y=top_corr.index,
    orientation="h",
    title="Top Features Correlated with Event"
)

fig.show()

# Survival Perspective Visualization

In [22]:
fig = px.ecdf(
    train,
    x="time_to_hit_hours",
    color="event",
    title="Empirical CDF of Time to Evacuation Hit"
)

fig.show()